# Tutorial 06a — Fused Attention, Unpacked (a companion)

> A companion to **`tutorials_jupyter/06-fused-attention.ipynb`**, OpenAI's Triton
> implementation of **FlashAttention-2**. That tutorial is *one code cell with ~760 lines* —
> eleven functions, four hardware-detection helpers, TMA descriptors, warp specialization,
> FP8, and a backward pass, all at once. This notebook slows it down: **one idea per section**,
> a figure, the exact lines from the tutorial, and a small exercise you can run.
>
> **Persona:** entry-to-intermediate practitioner (you know PyTorch and the basic GPU memory
> hierarchy; you are learning Triton). Same voice as course Chapters 16a–16d.

### How to use this notebook

The heavy Triton kernel targets **Hopper/Blackwell** GPUs (H100, B200) and their Tensor Memory
Accelerator. You almost certainly don't have one. So the plan is:

- **Understand it on the CPU.** Every core idea (online softmax, tiling, the causal trick, the
  backward gradient) is re-derived as a tiny **NumPy simulation** that runs anywhere and is checked
  against a reference. These carry the exercises.
- **Run a trimmed kernel on any CUDA GPU.** Section 7 has a simplified Triton forward that runs on a
  consumer card (e.g. an RTX 4080) — the tutorial's own kernel, minus the Hopper-only machinery.
- **Read the advanced parts as callouts.** TMA, warp specialization, FP8, autotuning (Section 10):
  what they are and why they exist, not something you'll execute here.

### Learning objectives

By the end you will be able to:

- Explain **online softmax** and why it lets attention run in one pass with `O(1)` state per row.
- Read **`_attn_fwd_inner`** line by line: the `qk` matmul, the base-2 `exp2` trick, the running-max
  rescale factor `alpha`, and the fused `p @ v` accumulate.
- Explain the **causal STAGE trick** (off-band vs on-band blocks) and why it splits the loop in two.
- Map **`_attn_fwd`** to "one program = one query block of one (batch, head)", including the
  `m += log2(l)` epilogue that stores the logsumexp for the backward pass.
- Explain the **backward pass**: the `D = rowsum(O ∘ dO)` trick that collapses the softmax Jacobian,
  and why FlashAttention-2 parallelizes `dK/dV` over key blocks but `dQ` over query blocks.
- Recognize what **TMA descriptors, warp specialization, FP8, and autotuning** buy you in production.

### Sources grounding this notebook

- The tutorial itself: `tutorials_jupyter/06-fused-attention.ipynb` (OpenAI Triton kernel team).
- Dao — [*FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*](https://arxiv.org/abs/2307.08691) (2023). The loop order and backward parallelization.
- Dao, Fu, Ermon, Rudra, Ré — [*FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*](https://arxiv.org/abs/2205.14135) (2022). Online softmax, tiling, recomputation.
- [Triton fused-attention tutorial docs](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) and the [`TensorDescriptor` / warp-specialization docs](https://triton-lang.org/main/getting-started/tutorials/gluon/warp-specialization.html).

> This notebook builds on the same material as course **Chapter 16b (Flash Attention from Scratch, in CUDA)**.
> That chapter writes the kernel in raw CUDA; here we read the **Triton** version. Same algorithm, different tool.


## 0. A map of the monolith

Before we zoom in, here is the whole 760-line cell at a glance. Everything in it is one of these
thirteen pieces. Keep this table open as a legend — each row points to the section that opens it.

| In the tutorial cell | What it is | Covered in |
|---|---|---|
| `is_hip` / `is_cuda` / `supports_host_descriptor` / `is_hopper` / `is_blackwell` | **Target detection** — pick code paths per GPU capability | §1, §10 |
| `_attn_fwd_inner` | **Forward inner loop** — online softmax over one K/V tile (the heart) | §4 |
| `_attn_fwd` | **Forward kernel** — one query block per program; calls the inner loop; epilogue | §6 |
| `configs`, `keep`, `prune_invalid_configs`, `_host_descriptor_pre_hook` | **Autotuning** — search `BLOCK_M/BLOCK_N/num_warps/num_stages` | §10 |
| `_maybe_make_tensor_desc` | **TMA descriptor** builder (Hopper+) | §10 |
| `_attn_bwd_preprocess` | Backward step 1: `D = rowsum(O ∘ dO)` | §8 |
| `_attn_bwd_dkdv` | Backward inner loop for `dK`, `dV` (grid over **key** blocks) | §8 |
| `_attn_bwd_dq` | Backward inner loop for `dQ` (grid over **query** blocks) | §8 |
| `_attn_bwd` | Backward kernel — orchestrates the two loops above | §8 |
| `_attention(torch.autograd.Function)` | **Wrapper** — wires `forward`/`backward` into PyTorch autograd | §9 |
| `test_op` | Correctness test vs a PyTorch reference | §1, §7 |
| `bench_flash_attention`, `configs` | TFLOPS benchmark vs `flash-attn` | §11 |

**The shape convention** used everywhere: tensors are `(Z, H, N_CTX, HEAD_DIM)` = (batch, heads,
sequence length, head dimension). The tutorial calls batch `Z` and heads `H`. One *program*
(one instance of the kernel) handles one block of `BLOCK_M` query rows for one `(z, h)` pair.


## 1. Setup and the reference we check against

Every fast attention kernel is only trustworthy if it matches plain attention **bit-for-similar-bit**.
The tutorial's `test_op` builds a reference with ordinary PyTorch:

```python
# from test_op in the tutorial
M = torch.tril(torch.ones((N_CTX, N_CTX), device=DEVICE))     # causal mask
p = torch.matmul(q, k.transpose(2, 3)) * sm_scale             # S = QKᵀ · scale
if causal:
    p[:, :, M == 0] = float("-inf")
p = torch.softmax(p.float(), dim=-1)                          # row-wise softmax
ref_out = torch.matmul(p, v).half()                           # O = P V
```

That is the definition of attention: score, mask, softmax, weight the values. FlashAttention computes
the **exact same thing** — it never approximates — it just never *materializes* the `(N_CTX, N_CTX)`
matrix `p`. We start on the CPU with NumPy so it runs here and now, then move to Triton in §7.


In [ ]:
import numpy as np

def attention_reference(Q, K, V, causal, sm_scale):
    """Plain attention, the thing FlashAttention must reproduce.

    Q, K, V: arrays of shape (N, d) for a single head. Returns O of shape (N, d).
    This materializes the full (N, N) score matrix — exactly what we want to avoid on a GPU,
    but perfect as ground truth here.
    """
    N = Q.shape[0]
    S = (Q @ K.T) * sm_scale                       # (N, N) scores
    if causal:
        row = np.arange(N)[:, None]
        col = np.arange(N)[None, :]
        S = np.where(row >= col, S, -np.inf)       # keys after the query are invisible
    S = S - S.max(axis=1, keepdims=True)           # numerical stability
    P = np.exp(S)
    P = P / P.sum(axis=1, keepdims=True)           # row-wise softmax
    return P @ V                                   # (N, d)

rng = np.random.default_rng(0)
N, d = 64, 16
Q = rng.standard_normal((N, d))
K = rng.standard_normal((N, d))
V = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
print("reference output shape:", attention_reference(Q, K, V, causal=True, sm_scale=sm_scale).shape)


## 2. The one idea that makes it all work: online softmax

Standard softmax needs the **whole row** at once: you must know the maximum and the sum of `exp`
before you can normalize any element. That forces you to compute and store the entire `(N, N)`
score matrix. FlashAttention refuses to do that. Instead it walks the row **one tile of keys at a
time**, carrying just three running values per query row:

- `m` — the running **max** of the scores seen so far,
- `l` — the running **sum** of `exp(score − m)`,
- `acc` — the running weighted sum of value vectors.

When a new tile arrives with a larger max, everything computed under the old max is off by a factor
of `exp(m_old − m_new)`. So you **rescale** the old `l` and `acc` by that factor `α` before adding
the new tile. That's the entire trick.

![Online softmax running state](../course/figures/fig_t06a_online_softmax.svg)

The figure processes one row's six scores `[1, 3, 2, 5, 4, 0]` in three tiles of two. The running
`(m, l)` is all the state that survives between tiles. The exercise below reproduces exactly those
numbers, then proves the one-pass result equals a one-shot softmax.


In [ ]:
# EXERCISE A — implement one online-softmax step.
# Given the running state (m, l, acc) and a new tile of scores + values, return the
# updated (m, l, acc). This is the CPU twin of the tutorial's _attn_fwd_inner body.
import numpy as np

def online_update(m, l, acc, s_tile, v_tile):
    """One online-softmax step.
    m:  running max (scalar float)         l:  running denominator (scalar float)
    acc: running weighted value sum, shape (d,)
    s_tile: new scores, shape (n,)         v_tile: new values, shape (n, d)
    Return the updated (m, l, acc). On the first call m is -inf, l is 0, acc is zeros.
    """
    raise NotImplementedError

# --- checks: sweep one row in tiles of 2 and compare to a one-shot softmax ---
scores = np.array([1.0, 3.0, 2.0, 5.0, 4.0, 0.0])
Vrows = np.random.default_rng(7).standard_normal((6, 4))
m, l, acc = -np.inf, 0.0, np.zeros(4)
running = []
for t in range(0, 6, 2):
    m, l, acc = online_update(m, l, acc, scores[t:t + 2], Vrows[t:t + 2])
    running.append((round(float(m), 3), round(float(l), 3)))

# the exact running (m, l) drawn in the figure:
assert running == [(3.0, 1.135), (5.0, 1.203), (5.0, 1.578)], running
# and the one-pass output equals the one-shot softmax output:
p = np.exp(scores - scores.max()); p /= p.sum()
assert np.allclose(acc / l, p @ Vrows, atol=1e-12)
print("online softmax reproduces the figure and matches one-shot softmax:", np.round(acc / l, 4))


<details>
<summary>▶ Show solution</summary>

```python
def online_update(m, l, acc, s_tile, v_tile):
    m_new = max(m, float(s_tile.max()))
    # rescale factor for everything accumulated under the old max (0 on the first tile)
    alpha = np.exp(m - m_new) if np.isfinite(m) else 0.0
    p = np.exp(s_tile - m_new)              # unnormalized weights for this tile
    l_new = l * alpha + p.sum()
    acc_new = acc * alpha + p @ v_tile
    return m_new, l_new, acc_new
```

</details>

Now scale that single step up to a full **tiled forward**: loop query blocks on the outside, key/value
tiles on the inside, keep `(m, l, acc)` per query row, and rescale on every tile. This is FlashAttention's
forward pass with nothing hardware-specific — and it matches the reference for both causal and
non-causal attention.

In [ ]:
def flash_forward_numpy(Q, K, V, causal, sm_scale, BLOCK_M=16, BLOCK_N=16):
    """Tiled online-softmax attention on the CPU — the algorithm the Triton kernel implements."""
    N, d = Q.shape
    O = np.zeros((N, d))
    for i0 in range(0, N, BLOCK_M):                      # outer loop: query blocks
        qi = Q[i0:i0 + BLOCK_M]
        bm = qi.shape[0]
        m_i = np.full(bm, -np.inf)
        l_i = np.zeros(bm)
        acc = np.zeros((bm, d))
        hi = min(i0 + bm, N) if causal else N           # causal: skip keys we can't see
        for j0 in range(0, hi, BLOCK_N):                # inner loop: key/value tiles
            kj, vj = K[j0:j0 + BLOCK_N], V[j0:j0 + BLOCK_N]
            bn = kj.shape[0]
            qk = (qi @ kj.T) * sm_scale
            if causal:
                rows = i0 + np.arange(bm)[:, None]
                cols = j0 + np.arange(bn)[None, :]
                qk = np.where(rows >= cols, qk, -np.inf)
            m_new = np.maximum(m_i, qk.max(axis=1))
            alpha = np.where(np.isfinite(m_i), np.exp(m_i - m_new), 0.0)
            p = np.exp(qk - m_new[:, None])
            l_i = l_i * alpha + p.sum(axis=1)
            acc = acc * alpha[:, None] + p @ vj
            m_i = m_new
        O[i0:i0 + bm] = acc / l_i[:, None]
    return O

for causal in (False, True):
    o_flash = flash_forward_numpy(Q, K, V, causal, sm_scale)
    o_ref = attention_reference(Q, K, V, causal, sm_scale)
    err = np.abs(o_flash - o_ref).max()
    assert np.allclose(o_flash, o_ref, atol=1e-10), (causal, err)
    print(f"causal={causal!s:5} | tiled online-softmax == full softmax  (max err {err:.1e})")


## 3. Triton in five minutes

Triton is a Python DSL where **you write the code for one program (one tile of work)** and Triton
compiles it to a GPU kernel. You never write threads; you write *block-level* array math and Triton
maps it onto warps. Here are the only primitives you need to read the tutorial:

| Triton | Meaning | NumPy analogue |
|---|---|---|
| `tl.program_id(0)` | which program am I? (my index along grid dim 0) | the loop variable, but parallel |
| `tl.arange(0, BLOCK)` | a compile-time vector `[0, 1, …, BLOCK-1]` | `np.arange(BLOCK)` |
| `tl.load(ptr + offsets, mask=…)` | read a tile from global memory (HBM) | fancy-indexed read |
| `tl.store(ptr + offsets, x, mask=…)` | write a tile back to HBM | fancy-indexed write |
| `tl.dot(a, b)` | tile matmul on Tensor Cores (fp32 accumulate) | `a @ b` |
| `tl.max`, `tl.sum` | row/col reductions | `x.max(axis)`, `x.sum(axis)` |
| `tl.math.exp2(x)` | base-2 exponential (fast HW instruction) | `np.exp2(x)` |
| `BLOCK_M: tl.constexpr` | a **compile-time** constant (specializes the kernel) | a Python constant |

The kernel is **launched over a grid**. For the forward pass the grid is:

```python
def grid(META):
    return (triton.cdiv(q.shape[2], META["BLOCK_M"]), q.shape[0] * q.shape[1], 1)
```

Read that as: **dim 0** = one program per block of `BLOCK_M` query rows; **dim 1** = one program per
`(batch, head)` pair. So a single program owns one `BLOCK_M × HEAD_DIM` slab of queries and streams all
the keys/values past it. Exercise B pins down that grid arithmetic.


In [ ]:
# EXERCISE B — reproduce the forward launch grid.
# Match the tutorial's grid lambda exactly:
#     (triton.cdiv(N_CTX, BLOCK_M), BATCH * N_HEADS, 1)
# where cdiv(a, b) is the ceiling division ceil(a / b).

def fwd_grid(batch, n_heads, n_ctx, block_m):
    """Return the 3-tuple grid (dim0, dim1, dim2) that _attn_fwd is launched with."""
    raise NotImplementedError

assert fwd_grid(4, 32, 4096, 128) == (32, 128, 1)     # 4096/128 = 32 query blocks; 4*32 = 128 (b,h) pairs
assert fwd_grid(1, 8, 1000, 128) == (8, 8, 1)         # cdiv(1000,128) = 8 (ceiling!), not 7
print("grid arithmetic OK — one program per (query block, batch-head)")


<details>
<summary>▶ Show solution</summary>

```python
def fwd_grid(batch, n_heads, n_ctx, block_m):
    cdiv = -(-n_ctx // block_m)          # ceiling division
    return (cdiv, batch * n_heads, 1)
```

</details>

## 4. `_attn_fwd_inner` — the heart

This is the function that runs online softmax over one K/V tile. Here is the core of it, verbatim
from the tutorial (the non-causal path):

```python
# from _attn_fwd_inner, for each K/V tile:
k = desc_k.load([offsetk_y, 0]).T
qk = tl.dot(q, k)                          # (BLOCK_M, BLOCK_N) scores for this tile
m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)   # new running max
qk = qk * qk_scale - m_ij[:, None]         # stabilized, scaled logits
p = tl.math.exp2(qk)                        # <-- exp2, not exp (see below)
alpha = tl.math.exp2(m_i - m_ij)            # rescale factor for the old accumulator
l_ij = tl.sum(p, 1)                         # this tile's contribution to the denominator
acc = acc * alpha[:, None]                  # rescale the running output...
v = desc_v.load([offsetv_y, 0])
p = p.to(dtype)
acc = tl.dot(p, v, acc)                     # ...then add p @ v  (fused multiply-accumulate)
l_i = l_i * alpha + l_ij                    # update denominator
m_i = m_ij                                  # update running max
```

Line for line this is your `online_update` from Exercise A — running max, `alpha` rescale,
accumulate `p @ v`, carry `(m, l)`. Two things look unfamiliar, and both are performance tricks.

![Inner loop dataflow](../course/figures/fig_t06a_inner_loop.svg)


**Trick 1 — `exp2` instead of `exp`, and the magic number `1.44269504`.** GPUs have a fast hardware
instruction for base-2 exponentials (`ex2`) but not for base-`e`. So the kernel converts once, up front:

$$e^{x} = 2^{x\,\log_2 e}, \qquad \log_2 e = 1.44269504\ldots$$

In the setup you'll see `qk_scale *= 1.44269504`. Folding `log₂e` into the scale means every later
`exp2(qk)` computes the correct `exp` **for free**, using the fast instruction. That is why the
epilogue stores `m += log2(l)` (base-2 logsumexp) rather than `ln`. Exercise C makes this exact.

**Trick 2 — `tl.dot(p, v, acc)` with three arguments.** The third argument is the accumulator: it
computes `acc + p @ v` in a single Tensor-Core instruction, keeping the running sum in registers
instead of round-tripping through memory. The rescale `acc = acc * alpha[:, None]` happens **just
before** this, so the accumulator is always on the current max's scale.


In [ ]:
# EXERCISE C — the exp2 trick. Implement a numerically-stable softmax using ONLY np.exp2,
# exactly the way the Triton kernel does: fold log2(e) into the scores, then use base-2 exp.

def softmax_base2(scores):
    """Row-vector softmax computed with exp2 only. Must equal the usual exp-based softmax."""
    raise NotImplementedError

s = np.array([1.0, 3.0, 2.0, 5.0, 4.0, 0.0])
ref = np.exp(s - s.max()); ref /= ref.sum()
assert np.allclose(softmax_base2(s), ref, atol=1e-12)
# and it must stay stable on large logits (the whole point of subtracting the max):
big = np.array([1000.0, 1001.0, 999.0])
assert np.isfinite(softmax_base2(big)).all()
print("exp2 softmax matches exp softmax and is numerically stable")


<details>
<summary>▶ Show solution</summary>

```python
LOG2E = 1.4426950408889634   # = 1 / ln(2) = log2(e); the tutorial writes 1.44269504

def softmax_base2(scores):
    scaled = scores * LOG2E                 # fold log2(e) into the logits, once
    scaled = scaled - scaled.max()          # stability shift (in base-2 units)
    p = np.exp2(scaled)                      # 2**(x*log2e) == e**x
    return p / p.sum()
```

</details>

## 5. The causal STAGE trick

For causal attention a query at position `i` may only see keys at positions `≤ i`. Tiling that naively
would mean applying a `tl.where` mask on **every** tile — but most tiles are either **fully visible**
(entirely below the diagonal) or **fully hidden** (entirely above it). Masking is only needed on the
**diagonal** tile. The tutorial exploits this by splitting the key sweep into stages:

![Causal STAGE regions](../course/figures/fig_t06a_causal_stages.svg)

Inside `_attn_fwd_inner`, the `STAGE` constant picks the key range and whether to mask:

```python
if STAGE == 1:                       # off-band: keys strictly before this query block
    lo, hi = 0, start_m * BLOCK_M    #   -> no mask needed, full speed
elif STAGE == 2:                     # on-band: the diagonal block itself
    lo, hi = start_m * BLOCK_M, (start_m + 1) * BLOCK_M
    lo = tl.multiple_of(lo, BLOCK_M)
else:                                # STAGE == 3, non-causal: every key
    lo, hi = 0, N_CTX
```

and only STAGE 2 pays for the mask:

```python
if STAGE == 2:
    mask = offs_m[:, None] >= (start_n + offs_n[None, :])
    qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)   # future keys -> -1e6 -> exp2 -> 0
    m_ij = tl.maximum(m_i, tl.max(qk, 1))
    qk -= m_ij[:, None]
```


**Try it — the same picture, but live.** Click a query block and toggle causal to watch the
**off-band / on-band / future** split (and the `stage_key_range(...)` values) update. *This renders when you
run the cell in VS Code or Jupyter; on a static viewer like GitHub, use the figure above.*

In [ ]:
# Interactive stage map for §5 — renders in Jupyter / VS Code (their output sandbox runs the
# embedded JavaScript). With no IPython frontend it degrades to a note; static viewers such as
# GitHub fall back to the SVG above.
try:
    from IPython.display import HTML, display
    display(HTML(r"""<div id='t06a-stage'>
<div class='t6-title'>Interactive — which keys does one query block actually scan?</div>
<div class='t6-lede'>Pick a query block, then flip causal on/off. Green = <b>off-band</b> (all visible, no mask), amber = <b>on-band</b> diagonal (masked), grey = <b>future</b> (skipped). <code>(lo, hi)</code> is just the key-index range of each stage.</div>
<div class='t6-controls'>
<div class='t6-group'><span class='t6-lab'>query block (start_m)</span><div class='t6-btns' id='t6-blocks'></div></div>
<div class='t6-group'><span class='t6-lab'>mode</span><div class='t6-btns'><button class='t6-b t6-mode t6-sel' data-mode='causal'>causal</button><button class='t6-b t6-mode' data-mode='full'>non-causal</button></div></div>
</div>
<div class='t6-body'>
<div class='t6-gridwrap'><div class='t6-grid' id='t6-grid'></div>
<div class='t6-legend'><span class='t6-lg'><span class='t6-sw off'></span>off-band · no mask</span><span class='t6-lg'><span class='t6-sw on'></span>on-band · visible</span><span class='t6-lg'><span class='t6-sw onm'></span>on-band · masked</span><span class='t6-lg'><span class='t6-sw fut'></span>future · skipped</span></div>
</div>
<div class='t6-readout' id='t6-readout'></div>
</div>
</div>
<style>
#t06a-stage{display:block;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;color:#1a2130;background:#ffffff;border:1px solid #d3dbe6;border-radius:14px;padding:18px 20px;max-width:900px;line-height:1.5;box-sizing:border-box}
#t06a-stage *{box-sizing:border-box}
#t06a-stage code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.9em;background:#eef2f7;padding:1px 4px;border-radius:4px;color:#1a2130}
#t06a-stage .t6-title{font-size:16px;font-weight:700;margin-bottom:4px}
#t06a-stage .t6-lede{font-size:13px;color:#5c6879;margin-bottom:14px}
#t06a-stage .t6-controls{display:flex;flex-wrap:wrap;gap:18px;margin-bottom:14px}
#t06a-stage .t6-lab{display:block;font-size:11px;letter-spacing:.04em;color:#5c6879;font-family:ui-monospace,Menlo,Consolas,monospace;margin-bottom:6px}
#t06a-stage .t6-btns{display:flex;flex-wrap:wrap;gap:6px}
#t06a-stage .t6-b{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;color:#1a2130;background:#eef2f7;border:1px solid #d3dbe6;border-radius:7px;padding:6px 10px;cursor:pointer}
#t06a-stage .t6-b:hover{border-color:#b3c0d1}
#t06a-stage .t6-b.t6-sel{background:#4f46e5;border-color:#4f46e5;color:#ffffff}
#t06a-stage .t6-body{display:flex;flex-wrap:wrap;gap:22px;align-items:flex-start}
#t06a-stage .t6-grid{display:grid;grid-template-columns:24px repeat(8,30px);gap:3px}
#t06a-stage .t6-hd{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:10px;color:#7c8aa0;display:flex;align-items:center;justify-content:center}
#t06a-stage .t6-c{width:30px;height:30px;border-radius:5px;border:1px solid transparent;display:flex;align-items:center;justify-content:center;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;font-weight:600;background:#eef2f7;color:transparent}
#t06a-stage .t6-c.gh{background:rgba(90,104,125,.08)}
#t06a-stage .t6-c.ghv{background:rgba(90,104,125,.08);color:#9aa6b6}
#t06a-stage .t6-c.off{background:rgba(13,148,136,.16);color:#0d9488;border-color:#0d9488}
#t06a-stage .t6-c.on{background:rgba(217,119,6,.18);color:#b45309;border-color:#d97706}
#t06a-stage .t6-c.onm{background:repeating-linear-gradient(45deg,rgba(217,119,6,.18) 0 4px,transparent 4px 8px);color:#9aa6b6;border:1px dashed #d97706}
#t06a-stage .t6-c.fut{background:rgba(124,138,160,.14);color:#9aa6b6}
#t06a-stage .t6-c.full{background:rgba(13,148,136,.16);color:#0d9488;border-color:#0d9488}
#t06a-stage .t6-legend{display:flex;flex-wrap:wrap;gap:6px 14px;margin-top:12px;max-width:300px}
#t06a-stage .t6-lg{display:flex;align-items:center;gap:6px;font-size:11.5px;color:#5c6879}
#t06a-stage .t6-sw{width:13px;height:13px;border-radius:3px;border:1px solid #d3dbe6}
#t06a-stage .t6-sw.off{background:rgba(13,148,136,.16);border-color:#0d9488}
#t06a-stage .t6-sw.on{background:rgba(217,119,6,.18);border-color:#d97706}
#t06a-stage .t6-sw.onm{background:repeating-linear-gradient(45deg,rgba(217,119,6,.18) 0 3px,transparent 3px 6px);border:1px dashed #d97706}
#t06a-stage .t6-sw.fut{background:rgba(124,138,160,.14);border-color:#7c8aa0}
#t06a-stage .t6-readout{flex:1;min-width:270px}
#t06a-stage .t6-fact{font-size:13px;margin:0 0 10px}
#t06a-stage .t6-code{background:#f4f6fa;border:1px solid #d3dbe6;border-radius:9px;padding:12px;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;line-height:1.7;white-space:pre;overflow-x:auto;color:#1a2130}
#t06a-stage .t6-off{color:#0d9488;font-weight:700}
#t06a-stage .t6-on{color:#b45309;font-weight:700}
#t06a-stage .t6-cmt{color:#7c8aa0}
#t06a-stage .t6-note{font-size:12px;color:#5c6879;border-left:3px solid #4f46e5;background:rgba(79,70,229,.08);padding:6px 10px;border-radius:0 6px 6px 0;margin-top:12px}
</style>
<script>
(function(){
  var root=document.getElementById('t06a-stage');
  if(!root||root.dataset.init)return; root.dataset.init='1';
  var N=8,BM=2,BN=2,nB=N/BM, state={block:2,mode:'causal'};
  var grid=root.querySelector('#t6-grid'), blocks=root.querySelector('#t6-blocks'), readout=root.querySelector('#t6-readout');
  for(var b=0;b<nB;b++){(function(b){
    var btn=document.createElement('button');
    btn.className='t6-b t6-blk'; btn.textContent='Q'+(b*BM)+'-'+(b*BM+BM-1); btn.setAttribute('data-b',b);
    btn.onclick=function(){state.block=b;render();}; blocks.appendChild(btn);})(b);}
  root.querySelectorAll('.t6-mode').forEach(function(btn){
    btn.onclick=function(){state.mode=btn.getAttribute('data-mode');render();};});
  function mkhd(t){var d=document.createElement('div');d.className='t6-hd';d.textContent=t;return d;}
  function render(){
    root.querySelectorAll('.t6-blk').forEach(function(x){x.classList.toggle('t6-sel',(+x.getAttribute('data-b'))===state.block);});
    root.querySelectorAll('.t6-mode').forEach(function(x){x.classList.toggle('t6-sel',x.getAttribute('data-mode')===state.mode);});
    var bS=state.block*BM, bE=(state.block+1)*BM, causal=state.mode==='causal';
    grid.innerHTML=''; grid.appendChild(mkhd(''));
    for(var c=0;c<N;c++) grid.appendChild(mkhd('K'+c));
    for(var r=0;r<N;r++){
      grid.appendChild(mkhd('Q'+r));
      var inB=(r>=bS&&r<bE);
      for(var c2=0;c2<N;c2++){
        var cell=document.createElement('div'); cell.className='t6-c'; var cls,mk;
        if(!causal){ if(inB){cls='full';mk='✓';} else {cls='ghv';mk='·';} }
        else{ var vis=(c2<=r);
          if(inB){ if(c2<bS){cls='off';mk='✓';} else if(c2<bE){cls=vis?'on':'onm';mk=vis?'✓':'·';} else {cls='fut';mk='';} }
          else{ cls=vis?'ghv':'gh'; mk=vis?'·':''; } }
        cell.classList.add(cls); cell.textContent=mk; grid.appendChild(cell);
      }
    }
    var sm=state.block;
    if(causal){
      var tiles=[]; for(var j=0;j<bS;j+=BN) tiles.push('['+j+','+(j+BN)+')');
      var offT=tiles.length?tiles.join(' '):'— none —';
      readout.innerHTML=
        '<p class="t6-fact">Query block <b>start_m = '+sm+'</b> owns query rows <b>['+bS+', '+bE+')</b>.</p>'+
        '<div class="t6-code"><span class="t6-cmt"># start_m='+sm+', block_m='+BM+', n_ctx='+N+'</span>\n'+
        '<span class="t6-off">stage 1</span>  stage_key_range(1,'+sm+','+BM+','+N+') = (0, '+bS+')   <span class="t6-cmt"># off-band  keys [0,'+bS+')  no mask</span>\n'+
        '<span class="t6-on">stage 2</span>  stage_key_range(2,'+sm+','+BM+','+N+') = ('+bS+', '+bE+')   <span class="t6-cmt"># on-band   keys ['+bS+','+bE+')  masked</span>\n\n'+
        '<span class="t6-cmt"># inner loop:  for j0 in range(lo, hi, BLOCK_N)</span>\n'+
        '<span class="t6-off">stage 1</span>  visits key tiles: '+offT+'\n'+
        '<span class="t6-on">stage 2</span>  visits key tiles: ['+bS+','+bE+')</div>'+
        '<p class="t6-note"><b>Tiling check:</b> off-band ends at '+bS+' = on-band start (no gap, no overlap); on-band ends at '+bE+' = the causal edge (start_m+1)&times;block_m. Everything past '+bE+' is future → never scanned.</p>';
    } else {
      readout.innerHTML=
        '<p class="t6-fact">Query block <b>start_m = '+sm+'</b>, rows <b>['+bS+', '+bE+')</b>. Non-causal: every query sees every key.</p>'+
        '<div class="t6-code"><span class="t6-cmt"># non-causal: one full sweep, no masking, no split</span>\n'+
        '<span class="t6-off">stage 3</span>  stage_key_range(3,'+sm+','+BM+','+N+') = (0, '+N+')   <span class="t6-cmt"># keys [0,'+N+')</span>\n\n'+
        '<span class="t6-cmt"># for j0 in range(0, '+N+', BLOCK_N)  — every key tile</span></div>'+
        '<p class="t6-note">With <code>causal=False</code> there is nothing to skip and nothing to mask — the off-band / on-band split collapses into one green sweep.</p>';
    }
  }
  render();
})();
</script>"""))
except ImportError:
    print("Interactive widget needs a Jupyter/IPython frontend — see the static figure above.")

**How the two STAGE numbers connect.** In the *outer* kernel `_attn_fwd`, the caller passes
`STAGE = 3` for causal and `STAGE = 1` for non-causal, then dispatches:

```python
if STAGE & 1:   # runs for BOTH causal(3) and non-causal(1)
    ... _attn_fwd_inner(..., 4 - STAGE, ...)   # causal -> stage 1 (off-band); non-causal -> stage 3 (all keys)
if STAGE & 2:   # runs ONLY for causal(3)
    ... _attn_fwd_inner(..., 2, ...)           # the diagonal on-band block, masked
```

So a non-causal call makes **one** inner sweep over all keys (stage 3, no mask). A causal call makes
**two**: the cheap off-band sweep (stage 1) followed by the masked diagonal sweep (stage 2). Exercise D
pins down those key ranges.


In [ ]:
# EXERCISE D — reproduce the key range each inner-loop STAGE scans, for a given query block.
# Mirror the (lo, hi) logic above. start_m is the query-block index; keys are indexed 0..N_CTX.

def stage_key_range(stage, start_m, block_m, n_ctx):
    """Return (lo, hi), the half-open range of key indices _attn_fwd_inner sweeps.
       stage == 1 -> off-band (keys strictly before this query block)
       stage == 2 -> on-band  (the diagonal block itself)
       stage == 3 -> full     (non-causal: all keys)
    """
    raise NotImplementedError

# query block 3, block size 128, sequence length 4096:
assert stage_key_range(1, 3, 128, 4096) == (0, 384)        # off-band: keys 0 .. 3*128
assert stage_key_range(2, 3, 128, 4096) == (384, 512)      # on-band: the diagonal block
assert stage_key_range(3, 3, 128, 4096) == (0, 4096)       # non-causal: everything
# the off-band + on-band ranges must exactly tile the causal region [0, (start_m+1)*block_m):
off, on = stage_key_range(1, 3, 128, 4096), stage_key_range(2, 3, 128, 4096)
assert off[1] == on[0] and on[1] == 4 * 128
print("STAGE key ranges OK — off-band and on-band tile the visible region with no overlap")


<details>
<summary>▶ Show solution</summary>

```python
def stage_key_range(stage, start_m, block_m, n_ctx):
    if stage == 1:
        return (0, start_m * block_m)
    elif stage == 2:
        return (start_m * block_m, (start_m + 1) * block_m)
    else:  # stage == 3
        return (0, n_ctx)
```

</details>

## 6. `_attn_fwd` — the program body and the epilogue

Now the outer kernel that each program runs. Stripped of the descriptor plumbing (§10), its skeleton is:

```python
start_m = tl.program_id(0)          # which block of queries
off_hz  = tl.program_id(1)          # which (batch, head) — flattened
off_z, off_h = off_hz // H, off_hz % H

# per-row running state, initialized for online softmax
m_i = tl.zeros([BLOCK_M], tl.float32) - float("inf")
l_i = tl.zeros([BLOCK_M], tl.float32) + 1.0
acc = tl.zeros([BLOCK_M, HEAD_DIM], tl.float32)

qk_scale = sm_scale * 1.44269504    # fold in log2(e) once
q = desc_q.load([qo_offset_y, 0])   # load THIS block's queries; they stay in SRAM the whole time

if STAGE & 1:                       # off-band (or all keys, if non-causal)
    acc, l_i, m_i = _attn_fwd_inner(..., 4 - STAGE, ...)
if STAGE & 2:                       # on-band diagonal (causal only)
    acc, l_i, m_i = _attn_fwd_inner(..., 2, ...)

# --- epilogue ---
m_i += tl.math.log2(l_i)            # store logsumexp (base 2) for the backward pass
acc = acc / l_i[:, None]            # finally normalize the output
tl.store(m_ptrs, m_i)              # save M (the logsumexp), one scalar per query row
desc_o.store([qo_offset_y, 0], acc.to(dtype))   # save O
```

Three things to notice:

1. **`q` is loaded once and stays in SRAM** for the whole key sweep. That is the "keep the working set
   on chip" idea — `q` is reused against every key tile without re-reading HBM.
2. **`l_i` starts at `1.0`, not `0.0`.** A tiny detail: it avoids a divide-by-zero on rows that end up
   fully masked, and washes out once real weights accumulate.
3. **The epilogue saves `M = m + log2(l)`** — the per-row **logsumexp**. The backward pass will need it
   to recompute `p` without ever having stored the `(N, N)` matrix. This is the single value that lets
   FlashAttention-2 skip storing the softmax and still get exact gradients.


## 7. Run it: a trimmed Triton forward on any CUDA GPU

The tutorial's kernel needs Hopper/Blackwell (its `TensorDescriptor`/TMA path). The kernel below is the
**same algorithm** written with plain `tl.load`/`tl.store` and computed offsets, so it runs on any CUDA
GPU — including a consumer RTX 4080. It is a single-loop version (it applies the mask on every tile
rather than splitting into STAGE 1/2), which is simpler to read and slightly slower — perfect for learning.

Run this cell **on a machine with a CUDA GPU and Triton installed** (`pip install triton torch`). Without
one it prints a notice and skips — the CPU sections above already proved the algorithm is correct.


In [ ]:
# Runnable on any CUDA GPU with Triton. Guarded so it no-ops cleanly elsewhere.
try:
    import torch
    import triton
    import triton.language as tl
    HAVE_GPU = torch.cuda.is_available()
except Exception as _e:                       # noqa: BLE001
    HAVE_GPU = False
    print("Triton/PyTorch/CUDA not available — skipping the live kernel. "
          "The NumPy sections above already verified the algorithm.")

if HAVE_GPU:
    @triton.jit
    def _fa_fwd_min(Q, K, V, Out, sm_scale,
                    stride_qz, stride_qh, stride_qm, stride_qk,
                    stride_kz, stride_kh, stride_kn, stride_kk,
                    stride_vz, stride_vh, stride_vn, stride_vk,
                    stride_oz, stride_oh, stride_om, stride_ok,
                    H, N_CTX,
                    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
                    HEAD_DIM: tl.constexpr, CAUSAL: tl.constexpr):
        start_m = tl.program_id(0)
        off_hz = tl.program_id(1)
        off_z, off_h = off_hz // H, off_hz % H
        q_base = Q + off_z * stride_qz + off_h * stride_qh
        k_base = K + off_z * stride_kz + off_h * stride_kh
        v_base = V + off_z * stride_vz + off_h * stride_vh
        o_base = Out + off_z * stride_oz + off_h * stride_oh

        offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_d = tl.arange(0, HEAD_DIM)
        q = tl.load(q_base + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk,
                    mask=offs_m[:, None] < N_CTX, other=0.0)

        m_i = tl.zeros([BLOCK_M], tl.float32) - float("inf")
        l_i = tl.zeros([BLOCK_M], tl.float32) + 1.0
        acc = tl.zeros([BLOCK_M, HEAD_DIM], tl.float32)
        qk_scale = sm_scale * 1.44269504                      # fold in log2(e)

        hi = (start_m + 1) * BLOCK_M if CAUSAL else N_CTX
        for start_n in range(0, hi, BLOCK_N):
            offs_n = start_n + tl.arange(0, BLOCK_N)
            k = tl.load(k_base + offs_n[None, :] * stride_kn + offs_d[:, None] * stride_kk,
                        mask=offs_n[None, :] < N_CTX, other=0.0)          # (HEAD_DIM, BLOCK_N)
            qk = tl.dot(q, k) * qk_scale
            valid = offs_n[None, :] < N_CTX
            if CAUSAL:
                valid = valid & (offs_m[:, None] >= offs_n[None, :])
            qk = tl.where(valid, qk, -1.0e6)
            m_ij = tl.maximum(m_i, tl.max(qk, 1))
            p = tl.math.exp2(qk - m_ij[:, None])
            alpha = tl.math.exp2(m_i - m_ij)
            l_i = l_i * alpha + tl.sum(p, 1)
            acc = acc * alpha[:, None]
            v = tl.load(v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vk,
                        mask=offs_n[:, None] < N_CTX, other=0.0)          # (BLOCK_N, HEAD_DIM)
            acc = tl.dot(p.to(v.dtype), v, acc)
            m_i = m_ij
        acc = acc / l_i[:, None]
        tl.store(o_base + offs_m[:, None] * stride_om + offs_d[None, :] * stride_ok,
                 acc.to(Out.dtype.element_ty), mask=offs_m[:, None] < N_CTX)

    def flash_attention_triton(q, k, v, causal, sm_scale):
        B, Hn, N, D = q.shape
        assert D in (16, 32, 64, 128)
        o = torch.empty_like(q)
        BLOCK_M = BLOCK_N = 64
        grid = (triton.cdiv(N, BLOCK_M), B * Hn, 1)
        _fa_fwd_min[grid](q, k, v, o, sm_scale,
                          *q.stride(), *k.stride(), *v.stride(), *o.stride(),
                          Hn, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, HEAD_DIM=D, CAUSAL=causal)
        return o

    torch.manual_seed(0)
    B, Hn, N, D = 2, 4, 256, 64
    q = torch.randn(B, Hn, N, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, Hn, N, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, Hn, N, D, device="cuda", dtype=torch.float16)
    scale = 1.0 / (D ** 0.5)
    for causal in (False, True):
        tri = flash_attention_triton(q, k, v, causal, scale)
        s = (q.float() @ k.float().transpose(-2, -1)) * scale
        if causal:
            m = torch.tril(torch.ones(N, N, device="cuda", dtype=torch.bool))
            s = s.masked_fill(~m, float("-inf"))
        ref = torch.softmax(s, dim=-1) @ v.float()
        err = (tri.float() - ref).abs().max().item()
        assert err < 2e-2, (causal, err)
        print(f"causal={causal!s:5} | Triton kernel matches PyTorch  (max err {err:.2e})")


**See it — the launch grid, live.** The kernel above turned `tl.program_id(0)` and
`tl.program_id(1)` into a base pointer in four moves. Click any cell to pick one program instance and watch
its `(start_m, off_hz)` become a `(batch, head)` memory slab and a `BLOCK_M`-row slice of queries — the
same `off_z, off_h = off_hz // H, off_hz % H` and `q_base = Q + off_z*stride_qz + off_h*stride_qh` you just
read. *This renders when you run the cell in VS Code or Jupyter; on a static viewer like GitHub the §7
kernel above is the reference.*

In [ ]:
# Interactive grid->pointer map for §7 — renders in Jupyter / VS Code (their output sandbox runs the
# embedded JavaScript). With no IPython frontend it degrades to a note; a static viewer such as GitHub
# shows only this code, so the kernel and prose above stand on their own there.
try:
    from IPython.display import HTML, display
    display(HTML(r"""<div id="kg">
<style>
  #kg *, #kg *::before, #kg *::after { box-sizing: border-box; }
  #kg {
    --bg: #f5f6f8; --panel: #ffffff; --panel-2: #f0f2f5; --ink: #1a2130;
    --muted: #5b6577; --line: #dde1e8; --line-strong: #c3cad6;
    --batch: #6b46c1; --batch-bg: #ece4fb; --head: #2b6cb0; --head-bg: #dbeafe;
    --qblock: #c05621; --qblock-bg: #feecdc; --slab: #2f855a; --slab-bg: #d5f2e1;
    --code-bg: #f4f6f9; --shadow: 0 1px 2px rgba(20,30,50,.06), 0 6px 20px rgba(20,30,50,.05);
    color: var(--ink); background: var(--bg);
    font-family: ui-sans-serif, system-ui, -apple-system, "Segoe UI", Roboto, sans-serif;
    line-height: 1.55; -webkit-font-smoothing: antialiased;
    padding: clamp(14px, 3vw, 26px); border-radius: 14px; border: 1px solid var(--line);
    max-width: 900px; margin: 0 auto;
  }
  @media (prefers-color-scheme: dark) {
    #kg {
      --bg: #0e131a; --panel: #161d27; --panel-2: #1c242f; --ink: #e6eaf1;
      --muted: #9aa6b8; --line: #29323f; --line-strong: #3a4552;
      --batch: #b794f6; --batch-bg: #2c2244; --head: #7fb3e8; --head-bg: #17304a;
      --qblock: #f0a56a; --qblock-bg: #402413; --slab: #6fce9a; --slab-bg: #143528;
      --code-bg: #10161e; --shadow: 0 1px 2px rgba(0,0,0,.3), 0 8px 24px rgba(0,0,0,.28);
    }
  }
  #kg .eyebrow { font-size: 11.5px; letter-spacing: .14em; text-transform: uppercase; color: var(--muted); font-weight: 600; margin: 0 0 5px; }
  #kg .kg-h1 { font-size: 20px; line-height: 1.15; margin: 0 0 8px; letter-spacing: -.01em; font-weight: 700; }
  #kg .lede { color: var(--muted); font-size: 14px; margin: 0 0 4px; max-width: 66ch; }
  #kg code, #kg .mono { font-family: ui-monospace, "SF Mono", Menlo, Consolas, monospace; }
  #kg .lede code { color: var(--ink); background: var(--code-bg); padding: .08em .35em; border-radius: 5px; font-size: .92em; }
  #kg .controls { display: flex; flex-wrap: wrap; gap: 9px 16px; align-items: center; background: var(--panel); border: 1px solid var(--line); border-radius: 11px; padding: 11px 14px; margin: 16px 0 18px; box-shadow: var(--shadow); }
  #kg .ctrl { display: flex; align-items: center; gap: 7px; }
  #kg .ctrl > label { font-size: 12.5px; color: var(--muted); font-weight: 600; }
  #kg .stepper { display: inline-flex; align-items: center; border: 1px solid var(--line-strong); border-radius: 8px; overflow: hidden; }
  #kg .stepper button { border: 0; background: var(--panel-2); color: var(--ink); width: 25px; height: 27px; font-size: 15px; line-height: 1; cursor: pointer; font-family: inherit; }
  #kg .stepper button:hover { background: var(--line); }
  #kg .stepper button:disabled { opacity: .35; cursor: not-allowed; }
  #kg .stepper .val { min-width: 38px; text-align: center; font-variant-numeric: tabular-nums; font-weight: 600; font-size: 13.5px; padding: 0 4px; }
  #kg .panel { background: var(--panel); border: 1px solid var(--line); border-radius: 13px; padding: 16px 16px 18px; margin-bottom: 16px; box-shadow: var(--shadow); }
  #kg .panel h3 { font-size: 12px; letter-spacing: .09em; text-transform: uppercase; color: var(--muted); margin: 0 0 3px; font-weight: 700; }
  #kg .panel .sub { font-size: 13px; color: var(--muted); margin: 0 0 12px; }
  #kg .panel .sub b { color: var(--ink); }
  #kg .step-n { display: inline-flex; align-items: center; justify-content: center; width: 19px; height: 19px; border-radius: 50%; background: var(--panel-2); border: 1px solid var(--line-strong); color: var(--muted); font-size: 11.5px; font-weight: 700; margin-right: 7px; vertical-align: 1px; }
  #kg .grid-wrap { overflow-x: auto; padding-bottom: 4px; }
  #kg table.grid { border-collapse: separate; border-spacing: 3px; margin: 0 auto; }
  #kg table.grid th { font-weight: 600; color: var(--muted); font-size: 11px; padding: 2px; white-space: nowrap; }
  #kg .batchhdr { color: var(--batch); font-weight: 700; font-size: 10.5px; letter-spacing: .04em; background: var(--batch-bg); border-radius: 6px; padding: 3px 0; }
  #kg .rowhdr { text-align: right; padding-right: 8px; font-variant-numeric: tabular-nums; }
  #kg .rowhdr .rng { display: block; font-size: 9.5px; color: var(--muted); font-family: ui-monospace, monospace; }
  #kg td.cell { width: 33px; height: 29px; border-radius: 6px; cursor: pointer; background: var(--panel-2); border: 1px solid var(--line); font-family: ui-monospace, monospace; font-size: 10px; color: var(--muted); text-align: center; vertical-align: middle; transition: transform .08s ease, background .12s ease; font-variant-numeric: tabular-nums; }
  #kg td.cell:hover { border-color: var(--line-strong); transform: translateY(-1px); }
  #kg td.cell.colsel { background: var(--head-bg); }
  #kg td.cell.rowsel { background: var(--qblock-bg); }
  #kg td.cell.sel { background: var(--ink); color: var(--bg); border-color: var(--ink); font-weight: 700; box-shadow: 0 0 0 2px var(--panel), 0 0 0 4px var(--ink); }
  #kg .axis-y { color: var(--qblock); font-size: 11.5px; font-weight: 600; }
  #kg .gridnote { font-size: 12px; color: var(--muted); margin-top: 11px; }
  #kg .flow { display: grid; grid-template-columns: 1fr; gap: 9px; }
  #kg .eqbox { display: flex; flex-wrap: wrap; align-items: baseline; gap: 6px 10px; background: var(--code-bg); border: 1px solid var(--line); border-radius: 10px; padding: 10px 13px; font-family: ui-monospace, monospace; font-size: 13.5px; }
  #kg .eqbox .lbl { font-family: ui-sans-serif, system-ui, sans-serif; font-size: 10.5px; text-transform: uppercase; letter-spacing: .08em; color: var(--muted); margin-right: 4px; }
  #kg .tag { font-weight: 700; padding: 1px 7px; border-radius: 5px; font-variant-numeric: tabular-nums; }
  #kg .tag.b { color: var(--batch); background: var(--batch-bg); }
  #kg .tag.h { color: var(--head); background: var(--head-bg); }
  #kg .tag.q { color: var(--qblock); background: var(--qblock-bg); }
  #kg .tag.s { color: var(--slab); background: var(--slab-bg); }
  #kg .dim { color: var(--muted); }
  #kg .strip-wrap { overflow-x: auto; padding: 4px 0 10px; }
  #kg .strip { display: flex; gap: 3px; min-width: min-content; }
  #kg .slab { flex: 0 0 auto; min-width: 60px; padding: 8px 6px; border-radius: 7px; border: 1px solid var(--line); background: var(--panel-2); text-align: center; font-family: ui-monospace, monospace; font-size: 10.5px; color: var(--muted); }
  #kg .slab .sname { font-weight: 700; color: var(--ink); font-size: 11px; }
  #kg .slab .soff { font-size: 9px; opacity: .8; }
  #kg .slab.on { background: var(--slab-bg); border-color: var(--slab); color: var(--slab); }
  #kg .slab.on .sname { color: var(--slab); }
  #kg .slabaxis { font-size: 11px; color: var(--muted); margin: 2px 0 8px; }
  #kg .matrix { border: 1px solid var(--line-strong); border-radius: 8px; overflow: hidden; max-width: 320px; }
  #kg .band { display: flex; align-items: center; justify-content: space-between; padding: 6px 12px; font-family: ui-monospace, monospace; font-size: 11px; color: var(--muted); border-bottom: 1px solid var(--line); }
  #kg .band:last-child { border-bottom: 0; }
  #kg .band.on { background: var(--qblock-bg); color: var(--qblock); font-weight: 700; }
  #kg .band .brow { font-variant-numeric: tabular-nums; }
  #kg .matlabel { font-size: 11px; color: var(--head); font-weight: 600; text-align: center; margin-bottom: 6px; }
  #kg .summary { background: linear-gradient(180deg, var(--panel), var(--panel-2)); border: 1px solid var(--line-strong); }
  #kg .summary .line { font-family: ui-monospace, monospace; font-size: 13px; line-height: 1.9; }
  #kg .summary .line + .line { margin-top: 2px; }
  #kg .cols { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
  @media (max-width: 680px) { #kg .cols { grid-template-columns: 1fr; } }
  #kg td.cell:focus-visible, #kg .stepper button:focus-visible { outline: 2px solid var(--head); outline-offset: 2px; }
  @media (prefers-reduced-motion: reduce) { #kg * { transition: none; } }
</style>

<p class="eyebrow">Triton &middot; Fused Attention</p>
<div class="kg-h1">One grid cell = one program instance</div>
<p class="lede">The kernel launches over a 2-D grid: <code>program_id(0)</code> = which query block, <code>program_id(1)</code> = which <b>(batch, head)</b>. Click any cell to watch that one instance turn its coordinates into a <b>base pointer</b> into the flat <code>(Z, H, N_CTX, HEAD_DIM)</code> buffer.</p>

<div class="controls" id="kg-controls"></div>

<div class="panel">
  <h3><span class="step-n">1</span>The launch grid &mdash; rows = query blocks, cols = (batch, head)</h3>
  <p class="sub">Every cell is one parallel program. Rows are <code>program_id(0) = start_m</code>; columns are the flattened <code>program_id(1) = off_hz</code>, grouped by batch.</p>
  <div class="grid-wrap"><div id="kg-grid"></div></div>
  <p class="gridnote" id="kg-gridnote"></p>
</div>

<div class="panel">
  <h3><span class="step-n">2</span>Un-flatten off_hz into (batch, head)</h3>
  <p class="sub">The grid gave one number for "which (batch, head)". Integer-divide and modulo by <b>H</b> to split it back apart &mdash; like recovering row &amp; column from a flattened 2-D index.</p>
  <div class="flow" id="kg-unflatten"></div>
</div>

<div class="cols">
  <div class="panel">
    <h3><span class="step-n">3</span>Flat memory &rarr; this head's slab</h3>
    <p class="sub">Q is 4-D but memory is 1-D. Skip whole batches, then whole heads, to land on this instance's private <span style="color:var(--slab);font-weight:700">slab</span>.</p>
    <p class="slabaxis" id="kg-slabaxis"></p>
    <div class="strip-wrap"><div class="strip" id="kg-strip"></div></div>
    <div class="flow" id="kg-ptr" style="margin-top:12px"></div>
  </div>
  <div class="panel">
    <h3><span class="step-n">4</span>Within the slab &rarr; your query rows</h3>
    <p class="sub"><code>program_id(0)</code> then picks which <code>BLOCK_M</code> rows of this <code>N_CTX x HEAD_DIM</code> matrix you own.</p>
    <p class="matlabel">HEAD_DIM &rarr;</p>
    <div class="matrix" id="kg-matrix"></div>
    <div class="flow" id="kg-offsm" style="margin-top:12px"></div>
  </div>
</div>

<div class="panel summary">
  <h3>Address, end to end</h3>
  <div id="kg-summary"></div>
</div>
</div>
<script>
(function () {
  var cfg = { Z: 2, H: 4, N_CTX: 1024, HEAD_DIM: 64, BLOCK_M: 128 };
  var state = { start_m: 3, off_hz: 5 };
  var N_OPTS = [256, 512, 1024, 2048], BM_OPTS = [32, 64, 128, 256];

  function nb(x) { return x.toLocaleString('en-US'); }
  function nBlocks() { return Math.ceil(cfg.N_CTX / cfg.BLOCK_M); }
  function nHZ() { return cfg.Z * cfg.H; }
  function strides() { return { sm: cfg.HEAD_DIM, sh: cfg.N_CTX * cfg.HEAD_DIM, sz: cfg.H * cfg.N_CTX * cfg.HEAD_DIM }; }
  function clampState() {
    if (state.start_m >= nBlocks()) state.start_m = nBlocks() - 1;
    if (state.off_hz >= nHZ()) state.off_hz = nHZ() - 1;
  }
  function el(id) { return document.getElementById(id); }

  function stepper(label, get, set, opts) {
    var wrap = document.createElement('div'); wrap.className = 'ctrl';
    var l = document.createElement('label'); l.textContent = label; wrap.appendChild(l);
    var s = document.createElement('div'); s.className = 'stepper';
    var minus = document.createElement('button'); minus.textContent = '−'; minus.setAttribute('aria-label', 'decrease ' + label);
    var val = document.createElement('span'); val.className = 'val';
    var plus = document.createElement('button'); plus.textContent = '+'; plus.setAttribute('aria-label', 'increase ' + label);
    function sync() { val.textContent = get(); var arr = opts(), i = arr.indexOf(get()); minus.disabled = (i <= 0); plus.disabled = (i >= arr.length - 1); }
    minus.onclick = function () { var arr = opts(), i = arr.indexOf(get()); if (i > 0) { set(arr[i - 1]); render(); } };
    plus.onclick = function () { var arr = opts(), i = arr.indexOf(get()); if (i < arr.length - 1) { set(arr[i + 1]); render(); } };
    s.appendChild(minus); s.appendChild(val); s.appendChild(plus); wrap.appendChild(s);
    wrap._sync = sync; return wrap;
  }
  var ctrls = [
    stepper('Z (batch)', function () { return cfg.Z; }, function (v) { cfg.Z = v; clampState(); }, function () { return [1, 2, 3, 4]; }),
    stepper('H (heads)', function () { return cfg.H; }, function (v) { cfg.H = v; clampState(); }, function () { return [1, 2, 4, 8]; }),
    stepper('N_CTX', function () { return cfg.N_CTX; }, function (v) { cfg.N_CTX = v; clampState(); }, function () { return N_OPTS; }),
    stepper('BLOCK_M', function () { return cfg.BLOCK_M; }, function (v) { cfg.BLOCK_M = v; clampState(); }, function () { return BM_OPTS; }),
    stepper('HEAD_DIM', function () { return cfg.HEAD_DIM; }, function (v) { cfg.HEAD_DIM = v; }, function () { return [32, 64, 128]; })
  ];
  var cControls = el('kg-controls');
  ctrls.forEach(function (c) { cControls.appendChild(c); });

  function buildGrid() {
    var nb_ = nBlocks(), Z = cfg.Z, H = cfg.H;
    var tbl = document.createElement('table'); tbl.className = 'grid';
    var tr0 = document.createElement('tr'); tr0.appendChild(document.createElement('th'));
    for (var z = 0; z < Z; z++) {
      var th = document.createElement('th'); th.colSpan = H;
      var d = document.createElement('div'); d.className = 'batchhdr'; d.textContent = 'batch ' + z;
      th.appendChild(d); tr0.appendChild(th);
    }
    tbl.appendChild(tr0);
    var tr1 = document.createElement('tr');
    var corner = document.createElement('th'); corner.className = 'axis-y'; corner.textContent = 'start_m'; tr1.appendChild(corner);
    for (var hz = 0; hz < Z * H; hz++) { var th2 = document.createElement('th'); th2.textContent = 'h' + (hz % H); tr1.appendChild(th2); }
    tbl.appendChild(tr1);
    for (var m = 0; m < nb_; m++) {
      var tr = document.createElement('tr');
      var rh = document.createElement('th'); rh.className = 'rowhdr';
      var lo = m * cfg.BLOCK_M, hi = Math.min((m + 1) * cfg.BLOCK_M, cfg.N_CTX);
      rh.innerHTML = 'block ' + m + '<span class="rng">[' + lo + ',' + hi + ')</span>';
      tr.appendChild(rh);
      for (var oh = 0; oh < Z * H; oh++) {
        var td = document.createElement('td'); td.className = 'cell';
        td.textContent = '(' + m + ',' + oh + ')';
        td.setAttribute('tabindex', '0'); td.setAttribute('role', 'button');
        if (oh === state.off_hz) td.className += ' colsel';
        if (m === state.start_m) td.className += ' rowsel';
        if (oh === state.off_hz && m === state.start_m) td.className = 'cell sel';
        (function (mm, ohh) {
          td.onclick = function () { state.start_m = mm; state.off_hz = ohh; render(); };
          td.onkeydown = function (e) { if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); state.start_m = mm; state.off_hz = ohh; render(); } };
        })(m, oh);
        tr.appendChild(td);
      }
      tbl.appendChild(tr);
    }
    return tbl;
  }

  function eqrow(label, html) { return '<div class="eqbox"><span class="lbl">' + label + '</span>' + html + '</div>'; }

  function render() {
    clampState();
    ctrls.forEach(function (c) { c._sync(); });
    var H = cfg.H, oh = state.off_hz, sm = state.start_m;
    var off_z = Math.floor(oh / H), off_h = oh % H;
    var st = strides();
    var q_base = off_z * st.sz + off_h * st.sh;
    var row_lo = sm * cfg.BLOCK_M, row_hi = Math.min((sm + 1) * cfg.BLOCK_M, cfg.N_CTX);
    var first_row = q_base + row_lo * st.sm;

    var g = el('kg-grid'); g.innerHTML = ''; g.appendChild(buildGrid());
    el('kg-gridnote').innerHTML = 'Grid shape = <b>(' + nBlocks() + ', ' + nHZ() + ', 1)</b> &rarr; <b>' + (nBlocks() * nHZ()) + '</b> programs run in parallel. You clicked <span class="tag q">start_m ' + sm + '</span> <span class="tag h">off_hz ' + oh + '</span>.';

    el('kg-unflatten').innerHTML =
      eqrow('program_id(0)', '<span class="tag q">start_m = ' + sm + '</span> <span class="dim">&mdash; my query block</span>') +
      eqrow('program_id(1)', '<span class="tag h">off_hz = ' + oh + '</span> <span class="dim">&mdash; my (batch, head), flattened</span>') +
      eqrow('off_z = off_hz // H', oh + ' // ' + H + ' = <span class="tag b">' + off_z + '</span> <span class="dim">&rarr; batch index</span>') +
      eqrow('off_h = off_hz % H', oh + ' % ' + H + ' = <span class="tag h">' + off_h + '</span> <span class="dim">&rarr; head index</span>');

    el('kg-slabaxis').innerHTML = 'Flat buffer Q &mdash; ' + nHZ() + ' slabs of <span class="mono">' + nb(st.sh) + '</span> elements (N_CTX x HEAD_DIM) each:';
    var strip = el('kg-strip'); strip.innerHTML = '';
    for (var i = 0; i < nHZ(); i++) {
      var zz = Math.floor(i / H), hh = i % H;
      var sl = document.createElement('div'); sl.className = 'slab' + (i === oh ? ' on' : '');
      sl.innerHTML = '<div class="sname">b' + zz + 'h' + hh + '</div><div class="soff">@' + nb(i * st.sh) + '</div>';
      strip.appendChild(sl);
    }
    el('kg-ptr').innerHTML =
      '<div class="eqbox"><span class="lbl">q_base</span>Q + off_z&middot;stride_qz + off_h&middot;stride_qh</div>' +
      '<div class="eqbox">Q + <span class="tag b">' + off_z + '</span>&middot;' + nb(st.sz) + ' + <span class="tag h">' + off_h + '</span>&middot;' + nb(st.sh) + ' = Q + <span class="tag s">' + nb(q_base) + '</span></div>';

    var mat = el('kg-matrix'); mat.innerHTML = '';
    for (var b = 0; b < nBlocks(); b++) {
      var blo = b * cfg.BLOCK_M, bhi = Math.min((b + 1) * cfg.BLOCK_M, cfg.N_CTX);
      var band = document.createElement('div'); band.className = 'band' + (b === sm ? ' on' : '');
      band.innerHTML = '<span>block ' + b + '</span><span class="brow">rows [' + blo + ', ' + bhi + ')</span>';
      mat.appendChild(band);
    }
    el('kg-offsm').innerHTML =
      '<div class="eqbox"><span class="lbl">offs_m</span>start_m&middot;BLOCK_M + arange(0, BLOCK_M)</div>' +
      '<div class="eqbox"><span class="tag q">' + sm + '</span>&middot;' + cfg.BLOCK_M + ' + [0..' + (cfg.BLOCK_M - 1) + '] = rows [<span class="tag q">' + row_lo + '</span>, ' + row_hi + ')</div>';

    el('kg-summary').innerHTML =
      '<div class="line">program <span class="tag q">(start_m=' + sm + '</span>, <span class="tag h">off_hz=' + oh + ')</span>  &rarr;  batch <span class="tag b">' + off_z + '</span>, head <span class="tag h">' + off_h + '</span>, query rows <span class="tag q">[' + row_lo + ', ' + row_hi + ')</span></div>' +
      '<div class="line"><span class="dim">head slab starts at element</span> <span class="tag s">' + nb(q_base) + '</span>  &middot;  <span class="dim">its first query row at</span> <span class="tag s">' + nb(first_row) + '</span> <span class="dim">(= q_base + ' + row_lo + '&middot;' + st.sm + ')</span></div>';
  }

  render();
})();
</script>"""))
except ImportError:
    print("Interactive widget needs a Jupyter/IPython frontend — see the §7 kernel above.")

## 8. The backward pass — the part most tutorials skip

Backprop through attention has to differentiate the softmax, whose Jacobian is a dense `(N, N)` object
per row. FlashAttention never forms it. Two ideas make that possible.

![Backward dataflow](../course/figures/fig_t06a_backward.svg)

**Idea 1 — recompute, don't store.** The forward saved only `O` and the per-row logsumexp `M`. The
backward **recomputes** `p = exp2(qk − M)` tile by tile (cheap: one extra matmul), instead of reading a
stored `(N, N)` matrix from HBM. That is the `_attn_bwd_preprocess`/`_attn_bwd_*` recomputation you see.

**Idea 2 — the `D = rowsum(O ∘ dO)` trick.** The softmax-Jacobian term collapses to a single scalar per
row. `_attn_bwd_preprocess` computes it once:

```python
# _attn_bwd_preprocess
o  = tl.load(...)                      # O
do = tl.load(...).to(tl.float32)       # upstream grad dO
delta = tl.sum(o * do, axis=1)         # D_i = sum_j O_ij dO_ij  = rowsum(O ∘ dO)
tl.store(Delta + ..., delta)
```

Then every score gradient is just `dS = P ∘ (dP − D)`, with `dP = dO Vᵀ`. Exercise E shows why that one
scalar `D` is algebraically the whole softmax Jacobian.


In [ ]:
# EXERCISE E — the collapsed softmax Jacobian.
# Implement the score gradient the way _attn_bwd does it, using D = rowsum(O ∘ dO):
#     dP = dO @ Vᵀ ,  D = rowsum(dO ∘ O) ,  dS = P ∘ (dP − D[:, None])
# and confirm it equals the textbook softmax-Jacobian gradient  dS = P ∘ (dP − rowsum(P ∘ dP)).

def attention_backward_ds(P, dO, V):
    """P: softmax probs (n, n).  dO: upstream grad of the output (n, d).  V: values (n, d).
    Return dS, the gradient w.r.t. the pre-softmax scores S. Use the O∘dO form (compute O = P @ V)."""
    raise NotImplementedError

rng = np.random.default_rng(1)
S = rng.standard_normal((5, 5))
P = np.exp(S - S.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
V = rng.standard_normal((5, 3))
dO = rng.standard_normal((5, 3))

dS = attention_backward_ds(P, dO, V)
dP = dO @ V.T
ref = P * (dP - (P * dP).sum(1, keepdims=True))      # textbook softmax Jacobian
assert np.allclose(dS, ref, atol=1e-12), np.abs(dS - ref).max()
print("D = rowsum(O∘dO) reproduces the full softmax Jacobian — one scalar per row is enough")


<details>
<summary>▶ Show solution</summary>

```python
def attention_backward_ds(P, dO, V):
    O = P @ V                                    # the same O the forward pass produced
    dP = dO @ V.T                                # (n, n)
    D = (dO * O).sum(1, keepdims=True)           # rowsum(O ∘ dO): one scalar per query row
    return P * (dP - D)
# Why it works: rowsum(P ∘ dP)_i = Σ_j P_ij (dO_i·V_j) = dO_i·(Σ_j P_ij V_j) = dO_i·O_i = D_i.
```

</details>

**Why two separate backward kernels?** FlashAttention-2's key insight is that `dK`/`dV` and `dQ` want
*different* parallelization axes:

- `_attn_bwd_dkdv` fixes a **key** block `(K_j, V_j)` and loops over query rows — so it parallelizes over
  **key** blocks. Each program accumulates a complete `dK_j`, `dV_j`.
- `_attn_bwd_dq` fixes a **query** block and loops over key columns — so it parallelizes over **query**
  blocks, accumulating a complete `dQ_i`.

Splitting this way means each gradient is written by exactly one program (no atomics, no cross-program
accumulation). It is the backward-pass half of the FlashAttention-2 parallelism story. Notice the tiny
scaling epilogues in the code — `dq *= LN2` and `dk *= sm_scale` — undoing the `log2(e)` and `sm_scale`
factors that were folded in earlier so the `exp2` path stayed cheap.


## 9. `_attention` — wiring it into PyTorch autograd

The kernels are stitched into a normal `torch.autograd.Function` so `attention(q, k, v, ...)` behaves
like any differentiable op. The essential parts:

```python
class _attention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, causal, sm_scale, warp_specialize=True):
        ...
        M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), ...)  # per-row logsumexp
        stage = 3 if causal else 1
        grid = lambda META: (triton.cdiv(q.shape[2], META["BLOCK_M"]), q.shape[0]*q.shape[1], 1)
        _attn_fwd[grid](sm_scale, M, q.shape[0], q.shape[1], q, k, v, o, N_CTX=q.shape[2], ...)
        ctx.save_for_backward(q, k, v, o, M)     # <-- note: saves O and M, NOT the (N,N) scores
        return o

    @staticmethod
    def backward(ctx, do):
        q, k, v, o, M = ctx.saved_tensors
        ...
        _attn_bwd_preprocess[pre_grid](o, do, delta, ...)   # D = rowsum(O ∘ dO)
        _attn_bwd[grid](q, arg_k, v, ctx.sm_scale, do, dq, dk, dv, M, delta, ...)
        return dq, dk, dv, None, None, None, None

attention = _attention.apply
```

The one line that captures the whole memory argument is **`ctx.save_for_backward(q, k, v, o, M)`**.
A naive implementation would stash the `(batch, heads, N, N)` attention matrix for the backward pass —
gigabytes at long sequence length. FlashAttention saves only `O` and the vector `M`, and pays a little
recompute in `backward` to rebuild `p`. Memory drops from `O(N²)` to `O(N)`. That trade — a bit more
compute to avoid HBM traffic — is the entire point of the algorithm.


## 10. The advanced machinery (read-only): TMA, warp specialization, FP8, autotuning

Everything so far is the algorithm. The remaining ~40% of the tutorial is **hardware-specific speed**,
gated behind capability checks (`supports_host_descriptor()`, `is_hopper()`, `is_blackwell()`). On an
Ada card (RTX 40-series) these paths are simply not taken. Here's what they are and why they exist —
you won't run them here, but you should recognize them.

> **Going to production.** To actually exercise this machinery you need an **H100 (Hopper, sm_90)** or
> **B200 (Blackwell, sm_100)**, recent Triton, and the input dtypes each path expects. On consumer GPUs
> the kernel still runs — it just falls back to the plain path you read in §6–7.


**TMA — Tensor Memory Accelerator (`TensorDescriptor`, `_maybe_make_tensor_desc`).** On Hopper/Blackwell
a dedicated hardware unit copies tiles between HBM and SRAM asynchronously, described by a
`TensorDescriptor` (shape/strides/block shape) instead of hand-computed pointer offsets. The tutorial
builds these descriptors and, in the kernel, `desc_k.load([...])` issues a TMA copy. On older GPUs the
wrapper skips descriptors (`desc_q = q`) and Triton uses ordinary loads — exactly what our §7 kernel does.

**Warp specialization (`warp_specialize=True`).** Normally every warp does the same work. Warp
specialization splits them into a **producer** group that issues TMA copies and a **consumer** group that
runs the Tensor-Core matmuls, handing tiles off through shared memory — so data movement and compute
overlap instead of taking turns. Triton exposes it via the `warp_specialize` flag; it is only profitable
on Hopper/Blackwell, which is why the tutorial enables it selectively (`enable_ws = ... is_blackwell() or (is_hopper() and not causal)`).

**FP8 (`float8_e5m2`, `FP8_OUTPUT`).** For inference-scale throughput the tutorial can run the matmuls in
8-bit floats. That needs the transposed-V handling and Blackwell's FP8 Tensor Cores you see guarded
throughout (`if dtype == tl.float8e5: ...`). FP8 is forward-only here — the backward pass asserts fp16.

**Autotuning (`@triton.autotune`, `configs`, `keep`, `prune_invalid_configs`).** There is no single best
tile size. The tutorial enumerates `BLOCK_M ∈ {64,128}`, `BLOCK_N ∈ {32,64,128}`, `num_warps ∈ {4,8}`,
`num_stages ∈ {2,3,4}`, prunes the invalid/slow combinations, benchmarks the survivors on the **first**
call for each `(N_CTX, HEAD_DIM, …)` key, and caches the winner. That first-call cost is why a fresh
kernel "warms up" before it's fast.


## 11. Reading the benchmark

The tutorial closes with a `triton.testing` benchmark that reports **TFLOP/s** against the reference
`flash-attn` package across sequence lengths. The FLOP accounting is worth reading once:

```python
flops_per_matmul = 2.0 * BATCH * H * N_CTX * N_CTX * HEAD_DIM
total_flops = 2 * flops_per_matmul        # two matmuls: S = QKᵀ  and  O = P V
if causal:
    total_flops *= 0.5                     # causal skips ~half the (query, key) pairs
if mode == "bwd":
    total_flops *= 2.5                     # backward ≈ 2.0x forward + 0.5x recomputation
return total_flops * 1e-12 / (ms * 1e-3)   # TFLOP/s
```

Two lessons hide in those constants. **`* 0.5` for causal**: the STAGE trick from §5 isn't just tidy —
skipping the upper triangle really is half the work. **`* 2.5` for backward**: the backward pass does
roughly 2× the matmuls of the forward, *plus* the 0.5× recompute of `p` — the compute we spend to avoid
storing the `(N, N)` matrix. You are watching the IO-vs-compute trade paid back in numbers.

Attention is **memory-bound**, not compute-bound, at these sizes; FlashAttention wins by moving less
data, and the benchmark's rising TFLOP/s curve with sequence length is that win made visible.


## 12. Recap — the whole cell, now legible

You've now opened every box in the 760-line cell:

| Concept | In the tutorial | You reproduced it in |
|---|---|---|
| Online softmax (running max, `alpha` rescale) | `_attn_fwd_inner` | Exercise A + `flash_forward_numpy` |
| `exp2` + `1.44269504` base-2 trick | `qk_scale *= 1.44269504`, `tl.math.exp2` | Exercise C |
| One program per (query block, batch-head) | the `grid` lambda | Exercise B |
| Causal off-band / on-band split | the `STAGE` branches | Exercise D |
| Logsumexp saved for backward | `m_i += tl.math.log2(l_i)` | §6 |
| Recompute-not-store + `D = rowsum(O ∘ dO)` | `_attn_bwd_preprocess`, `_attn_bwd_*` | Exercise E |
| `dK/dV` over keys, `dQ` over queries | `_attn_bwd_dkdv`, `_attn_bwd_dq` | §8 |
| Save only `O`, `M` (not the `N×N` scores) | `ctx.save_for_backward(q,k,v,o,M)` | §9 |
| TMA / warp-spec / FP8 / autotune | the capability-gated paths | §10 (read-only) |

**What you can now do:**

- Re-read `tutorials_jupyter/06-fused-attention.ipynb` top to bottom and know what every function is for.
- Run the §7 kernel on any CUDA GPU and extend it — e.g. add the STAGE 1/2 split, or a backward pass,
  using your `attention_backward_ds` as the reference.
- Explain to someone why FlashAttention is *exact* (not an approximation) and why it is nonetheless faster:
  it moves less memory, and on modern GPUs memory is the bottleneck.

**Where to go next:** course **Chapter 16b** writes this same kernel in raw CUDA (no Triton), and
**Chapters 16c/16d** cover FlashAttention-3 (Hopper asynchrony) and FlashAttention-4. The tutorial you
just unpacked is the bridge between "attention on paper" and "attention that trains real models."
